# H2: Is the primary delay cause independent of carrier?

Pre-registered: completed flights delayed 15+ minutes, January 2023 to December 2025. Carriers: the top 8 by number of such flights, all others grouped as OTHER. Causes: Carrier, Weather, NAS, Security, LateAircraft; TIE and NONE dropped and counted. Pearson chi-squared test of independence; effect size bias-corrected Cramér's V; adjusted standardized residuals, |r| > 3 notable. Fallback: if Security has expected counts below 5, merge it into NAS.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from delaystats import plotting, results
from delaystats import stats_helpers as sh
from delaystats.athena import run_query

ALPHA = 0.05  # pre-registered, two-sided (docs/ANALYSIS_PLAN.md)
SEED = 20261014  # fixed for every bootstrap
pd.set_option("display.float_format", "{:,.4f}".format)


def fmt_p(p):
    return "< 0.0001" if p < 1e-4 else f"{p:.4f}"


def table(rs):
    return pd.DataFrame([r.as_row() for r in rs]).set_index("test")


CAUSES = ["Carrier", "Weather", "NAS", "Security", "LateAircraft"]
# Cohen's thresholds for Cramer's V by df* = min(rows, columns) - 1
V_THRESHOLDS = {
    1: (0.10, 0.30, 0.50),
    2: (0.07, 0.21, 0.35),
    3: (0.06, 0.17, 0.29),
    4: (0.05, 0.15, 0.25),
}

## Contingency table

In [ ]:
counts = run_query("h2_cause_by_carrier")
delayed_by_carrier = (
    counts.groupby("reporting_airline")["flights"].sum().sort_values(ascending=False)
)
top8 = list(delayed_by_carrier.index[:8])
dropped = counts[counts["primary_delay_cause"].isin(["NONE", "TIE"])]
dropped_counts = dropped.groupby("primary_delay_cause")["flights"].sum()
print(f"flights delayed 15+ min: {counts['flights'].sum():,}")
print(f"dropped: {dropped_counts.to_dict()} ({dropped['flights'].sum():,} rows)")

kept = counts[~counts["primary_delay_cause"].isin(["NONE", "TIE"])].copy()
kept["carrier"] = np.where(kept["reporting_airline"].isin(top8), kept["reporting_airline"], "OTHER")
observed = kept.pivot_table(
    index="carrier", columns="primary_delay_cause", values="flights", aggfunc="sum", fill_value=0
)[CAUSES]
observed = observed.loc[top8 + ["OTHER"]]
observed

## Assumption check: expected counts

In [ ]:
first = sh.chi2_independence(observed, name="h2_chi2_independence", n_boot=2000, seed=SEED)
print(f"minimum expected count: {first.expected.to_numpy().min():.1f}")
merged = False
if first.expected["Security"].min() < 5:
    # pre-registered fallback; logged in docs/DEVIATIONS.md
    observed = observed.assign(NAS=observed["NAS"] + observed["Security"]).drop(columns="Security")
    merged = True
    print("Security had expected counts below 5: merged into NAS (pre-registered fallback)")
first.expected.round(1)

## Test, effect size and residuals

In [ ]:
chi = (
    sh.chi2_independence(observed, name="h2_chi2_independence", n_boot=2000, seed=SEED)
    if merged
    else first
)
results.save("H2", [chi.result])
plotting.residual_heatmap(
    chi.adjusted_residuals,
    "h2_adjusted_residuals",
    "H2: adjusted standardized residuals, carrier x primary delay cause",
)
print("notable cells (|r| > 3):", len(chi.notable_cells), "of", chi.adjusted_residuals.size)
display(table([chi.result]))
chi.adjusted_residuals.round(1)

## Interpretation

In [ ]:
r = chi.result
df_star = min(observed.shape) - 1
small, medium, large = V_THRESHOLDS[min(df_star, 4)]
v = r.effect_size
size = "negligible" if v < small else "small" if v < medium else "medium" if v < large else "large"
res = chi.adjusted_residuals
top_over = res.stack().sort_values(ascending=False).head(3)
top_under = res.stack().sort_values().head(3)
over = "; ".join(f"{c} {k} ({x:+.0f})" for (c, k), x in top_over.items())
under = "; ".join(f"{c} {k} ({x:+.0f})" for (c, k), x in top_under.items())
display(
    Markdown(f"""
Delay cause and carrier are {"not independent" if r.p <= ALPHA else "consistent with independence"}
(Pearson chi-squared({r.df:.0f}) = {r.statistic:,.0f}, p = {fmt_p(r.p)}, alpha = {ALPHA}, a single
pre-registered test so no correction; n = {int(observed.to_numpy().sum()):,} delayed flights).

- **How much it matters:** bias-corrected Cramér's V = {v:.3f}, 95% CI [{r.effect_ci[0]:.3f},
  {r.effect_ci[1]:.3f}]: a **{size}** association by Cohen's thresholds for df* = {df_star}
  ({small}, {medium}, {large}). With this many flights the p-value is guaranteed to be tiny; V is
  the number to read.
- **Where the association sits:** most over-represented cells: {over}. Most under-represented:
  {under}. {len(chi.notable_cells)} of {res.size} cells have |residual| > 3.
- **Data caveat:** causes are reported by the carriers themselves, so differences can reflect how a
  carrier attributes delays as well as what causes them.
- Rows dropped before the test: {int(dropped["flights"].sum()):,} (TIE and NONE).
  {"Security was merged into NAS (pre-registered fallback)." if merged else "No merge was needed: every expected count is at least 5."}
""")
)